In [36]:
import pyTMD

import my_lib.funcs

import util.coordinate_transforms

import Tides

import pandas as pd
import numpy as np
import datetime

In [9]:
# Set path to event files
evts_path = "/Users/sambrown04/Documents/SURF/Events/2013_2013Events2stas" 

# load events in
events_list = my_lib.funcs.load_evt(evts_path)

In [12]:
feat = my_lib.funcs.extract_event_features(events_list)

In [18]:
# first let us make a list of events with only gz stations
gz_df = pd.DataFrame(columns = ["station", "pre-slip_area", "slip_severity", "peak_time", "total_delta", "start_time"])

for i, event in enumerate(feat):

    for _,row in event.iterrows():

        # print(row['station'][0:2])
        if row['station'][0:2] == "gz":
            
            gz_df.loc[len(gz_df)] = row

In [20]:
# get average x and y cor and lat lon for the gz stations
# need a list of x cors and a list of y cors
x_cors = []
y_cors = []

pre = 'gz'
numbers = ['01','02','03','04','05','12','13','14','15','16','17','18'] # had to change as data frame did not have some of the stations?
    
for num in numbers:
    station = f"{pre}{num}"

    x_col = f"{station}x"
    y_col = f"{station}y"
    
    # loop through events to get first instance station is transmitting
    for i, event in enumerate(events_list):
        if not event[x_col].isna().any():# make sure location is transmitting
            # get first instance of coordinates
            x_cors.append(event.at[0, x_col]) 
            y_cors.append(event.at[0,y_col])
            break

In [24]:
avg_x_cor = sum(x_cors) / len(x_cors)
avg_y_cor = sum(y_cors) / len(y_cors)

In [27]:
# Setup model
# Suspect this line creating the model is causing the long runtime. We are defining it here and passing it into the tidal_elevation function

### USER DEFINED PATH TO TIDE MODEL ###
tide_dir = "/Users/sambrown04/Documents/SURF"
#######################################

tide_mod = "CATS2008-v2023"

tides = Tides.Tide(tide_mod, tide_dir)

model1 = pyTMD.io.model(tide_dir, format="netcdf").elevation(tide_mod)
constituents = pyTMD.io.OTIS.read_constants(
    model1.grid_file,
    model1.model_file,
    model1.projection,
    type=model1.type,
    grid=model1.format,
)

In [32]:
# function that takes the x_cors and y_cors and time and returns the tide.
import time

def get_tide_height(x_cor, y_cor, start_time):
    # just going to do like example for now and just pull first val from list for simplicity.
    days = 1
    spacing = 1

    HR_PER_DAY = 24
    MIN_PER_HR = 60

    dates_timeseries = []
    initial_time = datetime.datetime.strptime(start_time, "%Y-%m-%d %H:%M:%S")
    for i in range(days * HR_PER_DAY * MIN_PER_HR // spacing):  # 30 days * 24 hr/day * 60 min/hr * 1/10 calculations/min
        dates_timeseries.append(initial_time + datetime.timedelta(minutes=spacing * i))

    #convert to lon and lat
    lon, lat = util.coordinate_transforms.xy2ll(x_cor, y_cor)
    # print(lon, lat)

    
    tides = Tides.Tide(tide_mod, tide_dir)
    
    start_time = time.time()
    tide_results = tides.tidal_elevation(
        model1,
        constituents,
        [lon],
        [lat],
        dates_timeseries,
    ).data.T[0]
    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Elapsed time: {elapsed_time} seconds")
    
    return tide_results[0]
    
    
    
    #HR_PER_DAY * MIN_PER_HR // spacing
    

In [ ]:
tide_h = get_tide_height(avg_x_cor, avg_y_cor, '2013-05-19 07:37:15')
print(tide_h)